# M6 — SQL: Trabalhando com Múltiplas Tabelas


In [ ]:
# M6 — SQL: Trabalhando com Múltiplas Tabelas

import pandas as pd
import sqlite3

cliente = pd.DataFrame({
    'id_cliente':    [5, 1, 2, 4, 6],
    'nome':          ['jose','maria','valentina','joana','fernando'],
    'valor_compra':  [500.43, 150.70, 210.99, 1300.50, 86.55],
    'loja_cadastro': ['cea','riachuelo','zara','pontofrio','pontofrio']
})

transacoes = pd.DataFrame({
    'id_cliente':   [1, 2, 3, 1, 1, 3, 4, 5, 3],
    'id_transacao': [768805383,768805399,818770008,76856563,767573759,
                     818575758,764545534,76766789,959569],
    'valor_compra': [50.74,30.90,110.00,2000.90,15.70,2.99,50.74,10.00,1100.00],
    'id_loja':      ['magalu','giraffas','postoshell','magalu','subway',
                     'seveneleven','extra','subway','shopee']
})

# Carrega no SQLite (simula o Athena)
con = sqlite3.connect(':memory:')
cliente.to_sql('cliente', con, index=False, if_exists='replace')
transacoes.to_sql('transacoes', con, index=False, if_exists='replace')

print("Tabelas carregadas!")
print(f"\ncliente ({len(cliente)} linhas):")
print(cliente)
print(f"\ntransacoes ({len(transacoes)} linhas):")
print(transacoes)


# QUERY 1 — UNION

print("\n" + "="*55)
print("QUERY 1 — UNION: id_cliente de ambas as tabelas")
print("="*55)

query_1 = pd.read_sql_query("""
SELECT id_cliente FROM transacoes
UNION
SELECT id_cliente FROM cliente
""", con)

print(query_1)
query_1.to_csv('query_1.csv', index=False)
print("query_1.csv salvo")


# QUERY 2 — INNER JOIN

print("\n" + "="*55)
print("QUERY 2 — INNER JOIN: id_cliente + nome")
print("="*55)

query_2 = pd.read_sql_query("""
SELECT transacoes.id_cliente, cliente.nome
FROM transacoes
INNER JOIN cliente
ON transacoes.id_cliente = cliente.id_cliente
""", con)

print(query_2)
query_2.to_csv('query_2.csv', index=False)
print("query_2.csv salvo")


# QUERY 3 — CROSS JOIN

print("\n" + "="*55)
print("QUERY 3 — CROSS JOIN: todas as combinações")
print("="*55)

query_3 = pd.read_sql_query("""
SELECT *
FROM cliente
CROSS JOIN transacoes
""", con)

print(query_3)
print(f"\nTotal: {len(query_3)} linhas (5 clientes × 9 transações)")
query_3.to_csv('query_3.csv', index=False)
print("query_3.csv salvo")


# QUERY 4 — LEFT JOIN

print("\n" + "="*55)
print("QUERY 4 — LEFT JOIN: todas as transações + dados do cliente")
print("="*55)

query_4 = pd.read_sql_query("""
SELECT *
FROM transacoes
LEFT JOIN cliente
ON cliente.id_cliente = transacoes.id_cliente
""", con)

print(query_4)
print(f"\nTotal: {len(query_4)} linhas")
print("(cliente_id=3 aparece com NULL — não tem cadastro em cliente)")
query_4.to_csv('query_4.csv', index=False)
print("query_4.csv salvo")


# QUERY 5 — RIGHT JOIN
# (SQLite não suporta RIGHT JOIN nativamente,
#  simulamos invertendo a ordem com LEFT JOIN)

print("\n" + "="*55)
print("QUERY 5 — RIGHT JOIN: todos os clientes + transações")
print("="*55)

query_5 = pd.read_sql_query("""
SELECT transacoes.id_cliente,
       transacoes.id_transacao,
       transacoes.valor_compra,
       transacoes.id_loja,
       cliente.id_cliente   AS id_cliente_cad,
       cliente.nome,
       cliente.valor_compra AS valor_compra_cad,
       cliente.loja_cadastro
FROM cliente
LEFT JOIN transacoes
ON cliente.id_cliente = transacoes.id_cliente
""", con)

print(query_5)
print(f"\nTotal: {len(query_5)} linhas")
print("(fernando aparece com NULL nas colunas de transacoes — nunca comprou)")
query_5.to_csv('query_5.csv', index=False)
print("query_5.csv salvo")


# RESUMO FINAL

print("\n" + "="*55)
print("RESUMO — M6 SQL Múltiplas Tabelas")
print("="*55)
for i, q in enumerate([query_1,query_2,query_3,query_4,query_5], 1):
    print(f"  query_{i}.csv → {len(q):>4} linhas")



Tabelas carregadas!

cliente (5 linhas):
   id_cliente       nome  valor_compra loja_cadastro
0           5       jose        500.43           cea
1           1      maria        150.70     riachuelo
2           2  valentina        210.99          zara
3           4      joana       1300.50     pontofrio
4           6   fernando         86.55     pontofrio

transacoes (9 linhas):
   id_cliente  id_transacao  valor_compra      id_loja
0           1     768805383         50.74       magalu
1           2     768805399         30.90     giraffas
2           3     818770008        110.00   postoshell
3           1      76856563       2000.90       magalu
4           1     767573759         15.70       subway
5           3     818575758          2.99  seveneleven
6           4     764545534         50.74        extra
7           5      76766789         10.00       subway
8           3        959569       1100.00       shopee

QUERY 1 — UNION: id_cliente de ambas as tabelas
   id_cliente
0   